# Medical Chatbot — RAG pipeline trials

Step-by-step experiments for the medical chatbot: load PDFs, split into chunks,
embed with sentence-transformers, index into Pinecone, and answer questions with
an LLM using retrieval-augmented generation (RAG).

> Run from the repository root (or adjust paths) with a `.env` file containing
> `PINECONE_API_KEY` and `OPENAI_API_KEY`.

In [ ]:
import os

from dotenv import load_dotenv

# Work from the repository root so data/ and src/ resolve
if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")

load_dotenv()
assert os.environ.get("PINECONE_API_KEY"), "Set PINECONE_API_KEY in .env"
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

## 1. Load the PDF data

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader("data/", glob="*.pdf", loader_cls=PyPDFLoader)
extracted_docs = loader.load()
print(f"Loaded {len(extracted_docs)} pages")

## 2. Keep only minimal metadata

In [ ]:
from langchain.schema import Document

minimal_docs = [
    Document(
        page_content=doc.page_content,
        metadata={"source": doc.metadata.get("source", "")},
    )
    for doc in extracted_docs
]
minimal_docs[0].metadata if minimal_docs else "no docs"

## 3. Split into chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
text_chunks = text_splitter.split_documents(minimal_docs)
print(f"{len(text_chunks)} chunks")
print(text_chunks[0].page_content[:300] if text_chunks else "no chunks")

## 4. Embedding model

`all-MiniLM-L6-v2` produces 384-dimensional vectors — small, fast and good
enough for semantic search over medical text.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
query_vector = embeddings.embed_query("What is acne?")
print(f"Vector dimension: {len(query_vector)}")

## 5. Create the Pinecone index and upsert the chunks

In [ ]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "medical-chatbot"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
pc.describe_index(INDEX_NAME).status

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
)

## 6. Test similarity search

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retrieved = retriever.invoke("What is acne?")
for doc in retrieved:
    print(doc.page_content[:200], "\n---")

## 7. Full RAG chain with an LLM

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from src.prompt import system_prompt

chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("human", "{input}")]
)

question_answer_chain = create_stuff_documents_chain(chat_model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "What are the treatments for acne?"})
print(response["answer"])